In [ ]:
import re
from pathlib import Path

import numpy as np
import xarray as xr
import cmcrameri.cm as cmc
from IPython.display import Image, display

from pism_terra.ismip7.greenland import mass_balance as mb

ROOT = Path("~/base/pism-terra/2026_10_ml_century/output").expanduser()
OUTPUT = Path("figures/ml")
REFERENCE_YEAR = "1915"
XLIM = ("1915", "2015")
ORDER = ["GIS_GIS", "GIS_NO", "GIS_NE", "GIS_CE", "GIS_SE", "GIS_SW", "GIS_CW", "GIS_NW"]

# The terms of PISM's mass budget: their sum is tendency_of_ice_mass.
BUDGET = [
    "tendency_of_ice_mass_due_to_surface_mass_flux",
    "tendency_of_ice_mass_due_to_basal_mass_flux",
    "tendency_of_ice_mass_due_to_flow",
    "tendency_of_ice_mass_due_to_discharge",
    "tendency_of_ice_mass_due_to_conservation_error",
]
EXTRA = ["ice_mass_transport_across_grounding_line"]  # for the flux panels

BUDGET = [
    "tendency_of_ice_mass",
]
EXTRA = ["ice_mass_transport_across_grounding_line"]  # for the flux panels


def open_region_file(path):
    """One region_*.nc on (gcm_id, ssp_id, time, region): forcing as the GCM, resolution as the pathway."""
    name = Path(path).name
    resolution = re.search(r"_g(\d+m)_", name).group(1)
    forcing = re.search(r"_id_(.+?)_ml_", name).group(1)
    ds = xr.open_dataset(path).set_index(region="region_name")[BUDGET + EXTRA]
    # Monthly means to annual means, stamped at the start of the year like Mankoff's.
    return ds.resample(time="YS").mean().expand_dims(gcm_id=[forcing], ssp_id=[resolution])


sums = xr.combine_by_coords(
    [open_region_file(p) for p in sorted(ROOT.glob("scalar/region_g*_ml_century_*.nc"))],
    combine_attrs="drop_conflicts",
)
resolutions = sums.ssp_id.values

regions = mb.mass_balance_series(sums, variables=BUDGET, reference_year=REFERENCE_YEAR)
regions = regions.reindex(region=[r for r in ORDER if r in regions["region"].values])

mankoff = mb.load_mankoff(
    str(ROOT / "observations" / mb.DEFAULT_MANKOFF), reference_year=REFERENCE_YEAR, start="1840",
    variables=mb.MANKOFF_CUMULATIVE + mb.MANKOFF_FLUXES,
)

# Line colours for the resolutions, a dashed line for the OCX run.
colors = cmc.lipari(np.linspace(0.1, 0.8, len(resolutions)))
mb.SSP_COLORS.update(dict(zip(resolutions, colors)))
mb.GCM_STYLES.update({"OCX": "dashed"})

mb.plot_region(regions, mankoff, OUTPUT / "greenland_mass_balance.png", xlim=XLIM, sle_reference=REFERENCE_YEAR)
display(Image(filename=OUTPUT / "greenland_mass_balance.png"))
mb.plot_regions(regions, mankoff, OUTPUT / "regional_mass_balance.png", xlim=XLIM)
display(Image(filename=OUTPUT / "regional_mass_balance.png"))